# Prompt Engineering

<a target="_blank" href="https://colab.research.google.com/github/imamitjain/notebooks/blob/main/04-llm-and-transformers/05_prompt_engineering.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Objective:** Master prompt design patterns — zero-shot, few-shot, chain-of-thought, system prompts, and structured output — to get the best results from LLMs.

**Prerequisites:** RAG basics (notebook 04)

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q openai


In [ ]:
from openai import OpenAI
import json

# To use these examples, set your OpenAI API key:
# client = OpenAI(api_key="your-key-here")
#
# Or set the OPENAI_API_KEY environment variable and use:
# client = OpenAI()
#
# The examples below show the prompt patterns — you can run them
# once you've configured your API key.

## 1. Zero-Shot Prompting

In [ ]:
# Zero-shot: ask the model to perform a task with no examples

zero_shot_prompt = """Classify the following customer review as Positive, Negative, or Neutral.

Review: "The laptop arrived on time and works great, but the battery life is shorter than advertised."

Classification:"""

print("=== Zero-Shot Prompt ===")
print(zero_shot_prompt)
print("\n(Expected: Neutral or Mixed — no examples needed, the model infers the task)")

# To run with OpenAI:
# response = client.chat.completions.create(
#     model="gpt-4o-mini",
#     messages=[{"role": "user", "content": zero_shot_prompt}]
# )
# print(response.choices[0].message.content)

## 2. Few-Shot Prompting

In [ ]:
few_shot_prompt = """Classify customer reviews as Positive, Negative, or Neutral.

Review: "Absolutely love this phone! Best purchase I've made."
Classification: Positive

Review: "Broke after two days. Complete waste of money."
Classification: Negative

Review: "It's fine. Does what it's supposed to do."
Classification: Neutral

Review: "Great camera quality but the software is buggy and slow."
Classification:"""

print("=== Few-Shot Prompt ===")
print(few_shot_prompt)
print("\n(Expected: Mixed/Neutral — the examples teach the model the format and criteria)")

## 3. Chain-of-Thought (CoT) Prompting

In [ ]:
# Without CoT
simple_prompt = """What is 23 * 47 + 156 / 12?"""

# With CoT
cot_prompt = """What is 23 * 47 + 156 / 12?

Let's solve this step by step:
1. First, calculate 23 * 47
2. Then, calculate 156 / 12
3. Finally, add the results together"""

print("=== Without Chain-of-Thought ===")
print(simple_prompt)
print("\n=== With Chain-of-Thought ===")
print(cot_prompt)
print("\nStep-by-step solution:")
print("  1. 23 * 47 = 1081")
print("  2. 156 / 12 = 13")
print("  3. 1081 + 13 = 1094")
print("\nCoT reduces errors on multi-step reasoning tasks significantly.")

## 4. System Prompts and Role Setting

In [ ]:
messages_expert = [
    {"role": "system", "content": "You are a senior Python developer who gives concise, "
     "production-quality code with brief explanations. Always include error handling "
     "and type hints."},
    {"role": "user", "content": "Write a function to retry an HTTP request with exponential backoff."}
]

messages_teacher = [
    {"role": "system", "content": "You are a patient programming teacher for beginners. "
     "Explain concepts using simple analogies. Avoid jargon. Use short code examples."},
    {"role": "user", "content": "What is a function in Python?"}
]

print("=== Expert System Prompt ===")
for msg in messages_expert:
    print(f"  [{msg['role']}]: {msg['content'][:100]}...")

print("\n=== Teacher System Prompt ===")
for msg in messages_teacher:
    print(f"  [{msg['role']}]: {msg['content'][:100]}...")

print("\nSame question, different system prompts → very different responses")
print("The system prompt sets the persona, tone, and constraints")

## 5. Structured Output (JSON Mode)

In [ ]:
json_prompt = """Extract the following information from the text and return it as JSON:
- name (string)
- age (integer)
- skills (array of strings)
- experience_years (integer)

Text: "Sarah Chen is a 29-year-old software engineer with 6 years of experience. 
She's proficient in Python, Go, and Kubernetes."

JSON:"""

expected_output = {
    "name": "Sarah Chen",
    "age": 29,
    "skills": ["Python", "Go", "Kubernetes"],
    "experience_years": 6
}

print("=== Structured Output Prompt ===")
print(json_prompt)
print(f"\nExpected output:\n{json.dumps(expected_output, indent=2)}")

# With OpenAI, use response_format for guaranteed JSON:
# response = client.chat.completions.create(
#     model="gpt-4o-mini",
#     messages=[{"role": "user", "content": json_prompt}],
#     response_format={"type": "json_object"}
# )

## 6. Prompt Templates and Chaining

In [ ]:
# Reusable prompt templates
def classification_prompt(text, categories):
    cats = ", ".join(categories)
    return f"""Classify the following text into one of these categories: {cats}

Text: "{text}"

Category:"""

def summarize_prompt(text, max_words=50):
    return f"""Summarize the following text in {max_words} words or fewer.

Text: "{text}"

Summary:"""

def translate_prompt(text, target_lang):
    return f"""Translate the following text to {target_lang}. 
Return only the translation, no explanations.

Text: "{text}"

Translation:"""

# Prompt chaining: output of one becomes input of the next
text = "The new AI model achieves state-of-the-art performance on multiple benchmarks."

print("=== Prompt Chain ===")
print(f"Step 1 - Classify:")
print(f"  {classification_prompt(text, ['Technology', 'Business', 'Science', 'Sports'])}")
print(f"\nStep 2 - Summarize:")
print(f"  {summarize_prompt(text, 20)}")
print(f"\nStep 3 - Translate:")
print(f"  {translate_prompt(text, 'Spanish')}")
print(f"\nChaining connects these steps: classify → summarize → translate")

## Try It Yourself

1. Write a prompt that classifies customer support tickets into categories. Compare zero-shot vs few-shot accuracy on 10 example tickets.
2. Use chain-of-thought prompting to solve a multi-step math problem. Compare the answer with and without CoT.
3. Build a prompt chain: first extract key entities from a paragraph, then generate a summary, then translate it.

In [ ]:
# Your code here